In [1]:
import random
from collections.abc import Mapping

import httpx
import pandas as pd

In [2]:
REFERER = "https://quote.eastmoney.com/center/gridlist.html"

USER_AGENTS = [
    # Windows +Google Chrome
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.7922.75 Safari/537.36",
    # Iphone
    "Mozilla/5.0 (iPhone; CPU iPhone OS 18_6_2 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/18.6 Mobile/15E148 Safari/604.1",
    # Android
    "Mozilla/5.0 (Linux; Android 15; Pixel 9) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.7922.75 Mobile Safari/537.36",
]

request_headers = {
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "zh-CN,zh;q=0.9",
    "Referer":REFERER,
    "User-Agent": random.choice(USER_AGENTS)
}

In [28]:
client = httpx.Client(
  headers=request_headers,
  timeout=httpx.Timeout(connect=5.0, read=10.0, write=10.0, pool=10.0),
  # 设置最高连接数，防止误发大量并发，这样超量的会进行等待，超过pool timeout就报错
  limits=httpx.Limits(max_keepalive_connections=2, max_connections=2, keepalive_expiry=5.0),
  trust_env=True,
  follow_redirects=False,
)

## https://push2.eastmoney.com/api/qt/clist/get 接口分析
**GET** Method
- "http://push2.eastmoney.com/api/qt/clist/get" 报错 RemoteProtocolError: Server disconnected without sending a response.
- "http://push2delay.eastmoney.com/api/qt/clist/get" 成功（akshare方式）
- "http://pushguest.eastmoney.com/api/qt/clist/get" 开发者工具找到的，偶尔出现同1的错误，不要连发                                                                       
- 
### Request Params

该接口为分页接口，通过 `pn` 控制页码、`pz` 控制每页返回数量，拉取全量数据时从 `pn=1` 开始逐页递增，直到累计数量达到 `data.total`、当前页为空或返回数量小于 `pz`。`fields` 决定返回哪些字段，类似sql的select语法。`fid` 指定排序字段，`po=1` 为降序、`po=0` 为升，例如 `fid=f3` 直接获取涨幅靠前的记录，通常使用 `fid=f12` 保持分页顺序稳定。`np`为`1` 时 `data.diff` 为数组（recommended），`2` 时为以序号为键的对象。`fltt=1` 返回供网页格式化的缩放值，`fltt=2` 直接返回小数，程序化请求通常使用 `2`。`invt=2` 是语义未公开的行情兼容参数，参考原样保留。`ut` 是客户端标识而不是账号认证，固定值`fa5fd1943c7b386f172d6893dbfba10b
`。普通 JSON API 请求可省略后续参数：`cb` 用于 JSONP 包装，`dect`、`wbp2u` 是网页内部参数，`_` 是防缓存时间戳。

`fs` 用于筛选证券范围。表达式中的逗号 `,` 表示 OR，空格表示 AND；空格在 URL 查询字符串中通常编码为 `+`，所以 `m:1 t:2` 与 `m:1+t:2` 等价；`!` 表示排除。各标记及常见取值见下表：`m` 表示市场或数据源，`t` 表示该市场下的证券类型，`s` 表示更细的子类型，`b` 表示预定义集合，`f`、`e` 表示附加筛选，`i` 用于直接指定行情 ID。

| 标记 | 常见取值或写法 | 含义与示例 |
| --- | --- | --- |
| `m`：沪深与板块 | `0`、`1`、`2`、`90` | `m:0` 为深市数据源，北交所也使用它并叠加 `t:81 s:2048`；`m:1` 为沪市；`m:2` 为东财单独划分的中证系列指数行情源，不是第三个交易所；`m:90` 为板块数据源。 |
| `m`：海外证券与指数 | `105`、`106`、`107`、`116`、`124`、`125`、`128`、`153`、`155`、`156`、`305` | `105`、`106`、`107` 分别用于纳斯达克、纽交所和美股第三子市场；`116` 仅见于 efinance 的港股市场号映射，当前 `fs` 预设使用 `128`；`124`、`125`、`305` 为港股指数子市场，`128` 为港股证券，`153` 为美股 OTC/Pink/ADR 类市场，`155` 为伦交所，`156` 为伦交所 IOB 等国际证券子市场。 |
| `m`：外汇 | `119`、`120`、`133` | 分别为普通外汇交叉盘、人民币中间价和离岸人民币交叉盘；常用组合为 `m:119,m:120,m:133`。 |
| `m`：期货 | `8`、`113`、`114`、`115`、`142`、`225` | 分别为中金所、上期所、大商所、郑商所、上海国际能源交易中心和广期所。 |
| `m`：期权 | `10`、`12`、`140`、`141`、`151`、`163`、`226` | 分别为上交所、深交所、大商所、郑商所、上期所、上海国际能源交易中心和广期所期权。 |
| `t`：`m:0` 下的类型 | `5`、`6`、`7`、`10`、`13`、`80`、`81` | `t:5` 为深证指数，`t:6` 为深市 A 股，`t:7` 为深市 B 股，`t:10 e:97` 为深市 REITs，`t:13` 为部分资金流接口的兼容分支，`t:80` 为创业板，`t:81 s:2048` 为北交所证券。 |
| `t`：`m:1` 下的类型 | `1`、`2`、`3`、`9`、`23` | `t:1` 为上证指数，`t:2` 为沪市 A 股，`t:3` 为沪市 B 股，`t:9 e:97` 为沪市 REITs，`t:23` 为科创板。 |
| `t`：其他市场 | `m:90 t:1/2/3`、`m:128 t:1/2/3/4` | 板块数据源下 `t:1`、`t:2`、`t:3` 分别为地域、行业、概念板块；港股数据源下 `t:1`、`t:2`、`t:3`、`t:4` 分别用于 REIT/信托、债务证券、主板和创业板。 |
| `t`：英股内部分类 | `m:155 t:1/2/3`、`m:156 t:1/2/5/6/7/8` | AkShare 和 efinance 将这些内部上市分段合并查询；源码没有给出稳定的一一中文名称。 |
| `s` | `2`、`3`、`2048` | `m:1 s:2` 和部分预设中的 `m:1 s:3` 用于上证指数分支；`m:0 s:3` 为两网及退市；`m:0 t:81 s:2048` 为北交所。`s` 必须结合 `m`、`t` 解释。 |
| `b`：`MK` 集合 | `MK0010`、`MK0021`—`MK0024`、`MK0827`、`MK0201`、`MK0216`—`MK0221`、`MK0354`、`MK0356`、`MK0404`—`MK0407` | 分别用于重要指数、ETF 子集合、中概股、知名美股分类、可转债、交易所回购和 LOF。`MKxxxx` 是不可继续拆解的内部集合 ID，不是证券代码。 |
| `b`：`BK` / `DLMK` 集合 | `BK0498`、`BK0707`、`BK0804`、`BKxxxx`、`DLMK0101`、`DLMK0106`、`DLMK0144`、`DLMK0146` | `BK` 用于板块或资格池，如 A/B 股比价、沪股通、深股通及动态板块；`DLMK` 用于网页预设列表，如 AH 股比价、知名港股和港股通成分池。 |
| `f` | `4`、`8`、`!2`、`!50` | `f:4` 为风险警示板，`f:8` 为新股；`f:!2`、`f:!50` 分别是资金流和板块相关调用中使用的排除条件，服务端标记名称未公开。 |
| `e` | `97` | 额外类别条件；`m:1 t:9 e:97,m:0 t:10 e:97` 用于沪深 REITs。 |
| `i` | `i:<市场号>.<代码>` | 直接指定行情 ID，例如 `i:1.000001`、`i:0.399001`、`i:100.HSI`、`i:100.SPX`；多个 ID 用逗号合并。 |

`fields` 常用字段解析：

| 字段 | 含义 |
| --- | --- |
| `f1` / `f152` | 价格类 / 比例类的精度与显示辅助字段，主要供 `fltt=1` 的网页格式化使用，不是独立行情指标。 |
| `f2` / `f3` / `f4` | 最新价 / 涨跌幅 / 涨跌额。 |
| `f5` / `f6` | 成交量 / 成交额。 |
| `f7` / `f8` | 振幅 / 换手率。 |
| `f9` / `f10` / `f11` | 动态市盈率 / 量比 / 5 分钟涨跌幅。 |
| `f12` / `f13` / `f14` | 证券代码 / 市场编号 / 证券名称；可将 `f13` 和 `f12` 的值拼成统一行情 ID。 |
| `f15` / `f16` / `f17` / `f18` | 最高价 / 最低价 / 今开价 / 昨收价。 |
| `f19` / `f26` | 证券类型内部编码 / 上市日期。 |
| `f20` / `f21` | 总市值 / 流通市值。 |
| `f22` / `f23` | 3 分钟涨速 / 市净率。 |
| `f24` / `f25` | 60 日涨跌幅 / 年初至今涨跌幅。 |
| `f28` / `f31` / `f32` | 昨结价 / 买一价 / 卖一价，常用于期货、期权或盘口行情。 |
| `f62` / `f124` | 主力净流入 / 交易时间。 |


In [47]:
target = "https://push2delay.eastmoney.com/api/qt/clist/get"

ETF_PAGINATION_PARAMS = {
  "np":1,
  "po":1,
  "fltt":2,
  "invt":2,
  "pn":1,
  "pz":5,
  "ut": "fa5fd1943c7b386f172d6893dbfba10b",
  "fs": "b:MK0021,b:MK0022,b:MK0023,b:MK0024,b:MK0827",
  "fid": "f12",
  "fields": "f12,f13,f14"
}

In [48]:
resp = client.request("GET", target, params=ETF_PAGINATION_PARAMS)
resp_json = resp.json()
data = resp_json.get("data", None)

In [51]:
print(f"当前市场ETF基金总数 {data.get('total', "unknown")}")

if data.get("diff", None):
    df = pd.DataFrame(data.get("diff"))
    print(df)

当前市场ETF基金总数 1584
      f12  f13          f14
0  589990    1  科创综指ETF华泰柏瑞
1  589980    1  科创100ETF汇添富
2  589960    1  科创新能源ETF易方达
3  589950    1   科创100ETF富国
4  589900    1    科创综指ETF博时
